In [1]:
#Menyiapkan Dataset
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)

print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")
print(data_tugas11.head())

Dataset berhasil dibuat: 2500 baris
        nama_menu      cabang waktu_pesan  qty  harga_satuan  total_penjualan
0  Kentang Goreng    Semarang       Malam    1         30000     27750.163532
1  Kentang Goreng  Yogyakarta        Sore    3         20000     64216.160744
2  Kentang Goreng    Semarang       Siang    5         22000    108722.792592
3  Kentang Goreng  Yogyakarta        Pagi    2         20000     54483.205872
4      Roti Bakar    Magelang       Malam    2         30000     58157.944120


In [2]:
#Menyiapkan SparkSession
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Tugas11_KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions","4")

#Membaca dataframe dari file csv
df_penjualan = spark.read.csv("tugas11_penjualan.csv", header = True, inferSchema=True)
df_penjualan.show(10)

26/11/16 16:26:55 WARN Utils: Your hostname, LOQ-CEED resolves to a loopback address: 127.0.1.1; using 192.168.0.130 instead (on interface wlan0)
26/11/16 16:26:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 16:26:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
|     Americano|  Magelang|       Pagi|  1|       30000|32462.966390290698|
|    Cappuccino|  Magelang|      Malam|  2|       25000| 51846.84285104993|
|    Cappuccino|  Semarang|       Pagi|  5|       25000|114192.26376783242|
|     Kopi Susu|Yogyakarta|       Pagi|  1|       22000| 9697.180344260187|
|     Americano|  Magelang|       Pagi|  2|       15000|32316.051747436813|
+-----------

In [23]:
#A. Feature Engineering
##Mengubah kolom string menjadi numerik dengan StringIndexer
from pyspark.ml.feature import StringIndexer

#urutan index berdasarkan frekuensi kemunculan (paling sering muncul -> index 0, dst)
indexer_namaMenu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
indexer_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
indexer_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

#mempelajari daftar kategori dengan fit(), lalu transform() untuk menerapkan perubahan
df_indexed = indexer_namaMenu.fit(df_penjualan).transform(df_penjualan)
df_indexed = indexer_cabang.fit(df_indexed).transform(df_indexed)
df_indexed = indexer_waktu.fit(df_indexed).transform(df_indexed)

df_indexed.select("nama_menu","nama_menu_idx","cabang","cabang_idx","waktu_pesan","waktu_pesan_idx").show(10)

+--------------+-------------+----------+----------+-----------+---------------+
|     nama_menu|nama_menu_idx|    cabang|cabang_idx|waktu_pesan|waktu_pesan_idx|
+--------------+-------------+----------+----------+-----------+---------------+
|Kentang Goreng|          5.0|  Semarang|       2.0|      Malam|            1.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Sore|            3.0|
|Kentang Goreng|          5.0|  Semarang|       2.0|      Siang|            2.0|
|Kentang Goreng|          5.0|Yogyakarta|       1.0|       Pagi|            0.0|
|    Roti Bakar|          4.0|  Magelang|       0.0|      Malam|            1.0|
|     Americano|          0.0|  Magelang|       0.0|       Pagi|            0.0|
|    Cappuccino|          2.0|  Magelang|       0.0|      Malam|            1.0|
|    Cappuccino|          2.0|  Semarang|       2.0|       Pagi|            0.0|
|     Kopi Susu|          1.0|Yogyakarta|       1.0|       Pagi|            0.0|
|     Americano|          0.

In [49]:
#menggabungkan semua kolom menjadi satu kolom features dengan Vector Assembler
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols = ["nama_menu_idx","cabang_idx","waktu_pesan_idx","qty","harga_satuan"],
    outputCol = "features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features").show(5, truncate=False)


+-------------------------+
|features                 |
+-------------------------+
|[5.0,2.0,1.0,1.0,30000.0]|
|[5.0,1.0,3.0,3.0,20000.0]|
|[5.0,2.0,2.0,5.0,22000.0]|
|[5.0,1.0,0.0,2.0,20000.0]|
|[4.0,0.0,1.0,2.0,30000.0]|
+-------------------------+
only showing top 5 rows



In [50]:
##B. Train / Test Split & pelatihan model
#membagi data menjadi 75% untuk training model dan 25% untuk testing
train, test = df_final.randomSplit([0.75,0.25], seed=101)

print("Jumlah data training :", train.count())
print("Jumlah data testing :", test.count())

#melatih model linear regression dengan labelCol="total_penjualan"
from pyspark.ml.regression import LinearRegression

#mendefinisikan model
lr = LinearRegression(featuresCol = "features", labelCol = "total_penjualan")

#melatih model menggunakan data training dengan fit()
model = lr.fit(train)

print("Model telah dikultivasi")
print("Koefisien tiap fitur : ",model.coefficients)
print("Intercept : ",model.intercept)

Jumlah data training : 1839
Jumlah data testing : 661
Model telah dikultivasi
Koefisien tiap fitur :  [-123.21425598658806,279.2845246167347,339.638795323158,22046.663271025744,3.022920868160664]
Intercept :  -66903.17766485857


In [51]:
prediction = model.transform(test)
prediction.select("nama_menu","cabang","waktu_pesan","qty","harga_satuan",
                  "total_penjualan","prediction").show(10)

+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|19721.170446051627|31056.146105506938|
|Americano|Magelang|      Malam|  1|       25000|  21236.0956379149|31056.146105506938|
|Americano|Magelang|      Malam|  4|       22000|  85834.8564401191| 88127.37331410217|
|Americano|Magelang|      Malam|  4|       22000|102174.09100317833| 88127.37331410217|
|Americano|Magelang|      Malam|  4|       30000|113044.63431555482|112310.74025938749|
|Americano|Magelang|      Malam|  5|       22000|115810.29600214415|110174.03658512792|
|Americano|Magelang|      Malam|  5|       25000|140386.40273615732| 119242.7991896099|
|Americano|Magelang|       Pagi|  2|       30000| 73842.64420790318| 67877.77492201283|
|Americano|Magelang|       Pagi|

In [52]:
##C. Evaluasi Model
from pyspark.ml.evaluation import RegressionEvaluator
#RMSE (Root Mean Square Error) -- rata-rata kesalahan prediksi dalam satuan yang sama dengan target
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse= evaluator_rmse.evaluate(prediction)

#R-squared -- seberapa besar proporsi variasi data yang berhasil dijelaskan oleh model (0 = buruk, 1 = sempurna)
evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction",metricName="r2")
r2 = evaluator_r2.evaluate(prediction)

print(f"RMSE = Rp {rmse:,.0f}")
print(f"R^2 = {r2:.4f} ({r2*100:.1f}% variasi data berhasil dijelaskan dengan model")

RMSE = Rp 10,901
R^2 = 0.9107 (91.1% variasi data berhasil dijelaskan dengan model


In [53]:
#D. Eksperimen Perbandingan Fitur
#Membuat model kedua dengan hanya fitur qty dan harga_satuan
assembler2 = VectorAssembler(
    inputCols = ["qty","harga_satuan"],
    outputCol = "features2"
)

df_final2 = assembler2.transform(df_indexed)
df_final2.select("features2").show(5, truncate=False)


+-------------+
|features2    |
+-------------+
|[1.0,30000.0]|
|[3.0,20000.0]|
|[5.0,22000.0]|
|[2.0,20000.0]|
|[2.0,30000.0]|
+-------------+
only showing top 5 rows



In [54]:
#melatih model
#membagi data menjadi 75% untuk training model dan 25% untuk testing
train, test = df_final2.randomSplit([0.75,0.25], seed=101)

print("Jumlah data training :", train.count())
print("Jumlah data testing :", test.count())

#melatih model linear regression dengan labelCol="total_penjualan"
#mendefinisikan model
lr = LinearRegression(featuresCol = "features2", labelCol = "total_penjualan")

#melatih model menggunakan data training dengan fit()
model2 = lr.fit(train)

print("Model telah dikultivasi")
print("Koefisien tiap fitur : ",model2.coefficients)
print("Intercept : ",model2.intercept)

Jumlah data training : 1839
Jumlah data testing : 661
Model telah dikultivasi
Koefisien tiap fitur :  [22048.50478817013,3.0210498099882637]
Intercept :  -66397.03118601868


In [66]:
prediction2 = model2.transform(test)
prediction2.select("nama_menu","cabang","waktu_pesan","qty","harga_satuan",
                  "total_penjualan","prediction").show(10)

+---------+--------+-----------+---+------------+------------------+------------------+
|nama_menu|  cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|        prediction|
+---------+--------+-----------+---+------------+------------------+------------------+
|Americano|Magelang|      Malam|  1|       25000|19721.170446051627|31177.718851858037|
|Americano|Magelang|      Malam|  1|       25000|  21236.0956379149|31177.718851858037|
|Americano|Magelang|      Malam|  4|       22000|  85834.8564401191| 88260.08378640363|
|Americano|Magelang|      Malam|  4|       22000|102174.09100317833| 88260.08378640363|
|Americano|Magelang|      Malam|  4|       30000|113044.63431555482|112428.48226630974|
|Americano|Magelang|      Malam|  5|       22000|115810.29600214415|110308.58857457376|
|Americano|Magelang|      Malam|  5|       25000|140386.40273615732|119371.73800453857|
|Americano|Magelang|       Pagi|  2|       30000| 73842.64420790318| 68331.47268996948|
|Americano|Magelang|       Pagi|

In [56]:
evaluator_rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(prediction2)

evaluator_r2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction",metricName="r2")
r2 = evaluator_r2.evaluate(prediction2)
print(f"RMSE = Rp {rmse:,.0f}")
print(f"R^2 = {r2:.4f} ({r2*100:.1f}% variasi data berhasil dijelaskan dengan model")

RMSE = Rp 10,896
R^2 = 0.9108 (91.1% variasi data berhasil dijelaskan dengan model


In [60]:
##E. Prediksi Skenario Baru
#Membuat 3 record skenario pesanan baru
data_skenario_baru = pd.DataFrame({
    "nama_menu": ["Kopi Susu", "Americano", "Roti Bakar"],
    "cabang": ["Magelang", "Yogyakarta", "Semarang"],
    "waktu_pesan": ["Pagi", "Siang", "Malam"],
    "qty": [2, 4, 3],
    "harga_satuan": [20000, 25000, 30000]
})
df_skenario = spark.createDataFrame(data_skenario_baru)
df.show(10)

+----------+----------+-----------+---+------------+
| nama_menu|    cabang|waktu_pesan|qty|harga_satuan|
+----------+----------+-----------+---+------------+
| Kopi Susu|  Magelang|       Pagi|  2|       20000|
| Americano|Yogyakarta|      Siang|  4|       25000|
|Roti Bakar|  Semarang|      Malam|  3|       30000|
+----------+----------+-----------+---+------------+



In [65]:
#Memasukan dataset baru langsung ke dataset yang sudah diprediksi sebelumnya
df_skenario_indexed = indexer_namaMenu.fit(df_penjualan).transform(df_skenario)
df_skenario_indexed = indexer_cabang.fit(df_penjualan).transform(df_skenario_indexed)
df_skenario_indexed = indexer_waktu.fit(df_penjualan).transform(df_skenario_indexed)

# Transformasi menggunakan VectorAssembler utama
df_skenario_final = assembler.transform(df_skenario_indexed)

# Melakukan prediksi menggunakan model regresi utama
prediksi_skenario = model.transform(df_skenario_final)

# Menampilkan hasil prediksi
prediksi_skenario.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "harga_satuan", "prediction"
).show(truncate=False)

+----------+----------+-----------+---+------------+-----------------+
|nama_menu |cabang    |waktu_pesan|qty|harga_satuan|prediction       |
+----------+----------+-----------+---+------------+-----------------+
|Kopi Susu |Magelang  |Pagi       |2  |20000       |37525.35198441961|
|Americano |Yogyakarta|Siang      |4  |25000       |97815.05923852405|
|Roti Bakar|Semarang  |Malam      |3  |30000       |90329.78901364886|
+----------+----------+-----------+---+------------+-----------------+



In [67]:
#Menghentikan Spark
spark.stop()
print("Spark berhenti")

Spark berhenti
